# 03 · Cắt đoạn segment và cache logits

Classifier nhận đúng 18 feature liên tiếp. Với mỗi segment nhãn `(start, end)` cần chọn **18 feature nào** — đây là chỗ phát hiện **lỗi crop**:
code gốc rút vị trí ngẫu nhiên mỗi lần đánh giá, nên kết quả test dao động giữa các lần chạy. Ở đây vị trí được cố định bằng tham số `u`.

Kết quả mức đoạn: baseline đạt 26.3 với crop cố định, chỉ 24.7 với crop ngẫu nhiên.

In [1]:
from __future__ import annotations
import sys
from pathlib import Path

# Notebook nằm trong thư mục notebooks/ -> thêm thư mục gốc repo vào sys.path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)

Repo root: <repo>


**`tcfall/segments.py` · phần import và hằng số**

- `WINDOW_DUR` ≈ 2.133 s: độ dài 1 cửa sổ VideoMAE.
- `SEQ_SPAN`: khoảng cách giữa tâm feature đầu và cuối trong chuỗi 18 feature.

In [2]:
from __future__ import annotations
import numpy as np
from tcfall.config import FEATURE_STRIDE, NUM_FEATURES, TARGET_FPS, WINDOW_FRAMES
WINDOW_DUR = WINDOW_FRAMES / TARGET_FPS                     # 2.133 s
SEQ_SPAN = (NUM_FEATURES - 1) * FEATURE_STRIDE              # khoảng cách tâm feature đầu-cuối

**`tcfall/segments.py` · _idx_at**

Đổi thời điểm `t` (giây) thành chỉ số feature có **tâm cửa sổ** gần `t` nhất.

In [3]:
def _idx_at(t: float) -> int:
    return max(0, round((t - WINDOW_DUR / 2) / FEATURE_STRIDE))

**`tcfall/segments.py` · window_start**

Chỉ số feature bắt đầu của chuỗi 18 feature cho một segment:
- Segment **dài hơn** chuỗi: trượt trong segment, `u = 0` là đầu, `u = 1` là cuối.
- Segment **ngắn hơn**: căn giữa segment trong chuỗi, dịch nhẹ theo `(2u − 1)`.
- Kẹp lại để không vượt quá số feature của video.

In [4]:
def window_start(start: float, end: float, u: float, n_feat: int) -> int:
    dur = end - start
    if SEQ_SPAN <= dur:
        idx = _idx_at(start + u * (dur - SEQ_SPAN))
    else:
        margin = SEQ_SPAN - dur
        ideal = start - margin / 2
        jitter = min(0.25 * dur, margin / 2)
        idx = max(0, _idx_at(ideal + (2 * u - 1) * jitter))
    end_idx = idx + NUM_FEATURES
    if end_idx > n_feat:
        idx = max(0, n_feat - NUM_FEATURES)
    if idx >= n_feat:
        idx = max(0, n_feat - 1)
    return idx

**`tcfall/segments.py` · gather**

Lấy 18 feature từ chỉ số đã chọn; nếu cuối video thiếu feature thì lặp feature cuối (giống code gốc).

In [5]:
def gather(feats: np.ndarray, idx: int) -> np.ndarray:
    """18 feature từ idx, lặp feature cuối nếu thiếu (giống code gốc)."""
    seq = feats[idx:idx + NUM_FEATURES]
    if len(seq) < NUM_FEATURES:
        seq = np.concatenate([seq, np.repeat(seq[-1:], NUM_FEATURES - len(seq), 0)], 0)
    return seq

**`tcfall/segments.py` · crop_positions**

`k = 1` → chỉ lát giữa (`u = 0.5`); `k > 1` → `k` lát đều nhau từ 0 đến 1, dùng cho **TTA thời gian** (trung bình dự đoán nhiều lát).

In [6]:
def crop_positions(k: int) -> np.ndarray:
    return np.array([0.5]) if k == 1 else np.linspace(0.0, 1.0, k)

**Ví dụ nhỏ:** vị trí chuỗi 18 feature cho segment dài và segment ngắn.

In [7]:
import numpy as np
n_feat = 300
for s, e in [(10.0, 16.0), (10.0, 10.8)]:
    pos = [window_start(s, e, u, n_feat) for u in crop_positions(5)]
    print(f"segment {s}-{e}s ->", pos)
feats = np.random.randn(20, 768)
print(gather(feats, 10).shape)  # vẫn đủ 18 dù chỉ còn 10 feature

segment 10.0-16.0s -> [67, 74, 81, 88, 95]
segment 10.0-10.8s -> [60, 61, 62, 62, 63]
(18, 768)


**`pipeline/s03_cache_logits.py` · phần import và hằng số**

`N_CROPS = 9` vị trí cố định, `N_RANDOM = 10` vị trí ngẫu nhiên (seed cố định) để đo độ dao động của cách đánh giá ngẫu nhiên cũ.

In [8]:
from __future__ import annotations
import pickle
import sys
from pathlib import Path
sys.path.insert(0, str(Path(ROOT / "pipeline/s03_cache_logits.py").resolve().parent.parent))
import numpy as np
import torch
from tqdm import tqdm
from tcfall.baseline import load_baseline, predict_logits
from tcfall.config import FEATURE_VARIANTS, OUTPUTS, feature_dir
from tcfall.data import clean_labels, load_labels, load_split
from tcfall.segments import crop_positions, gather, window_start
N_CROPS, N_RANDOM = 9, 10

**`pipeline/s03_cache_logits.py` · main**

Với mỗi biến thể feature và mỗi split của OOPS:
1. nạp nhãn đã làm sạch và gán split;
2. với mỗi segment, tạo 9 chuỗi ở vị trí cố định và 10 chuỗi ở vị trí ngẫu nhiên (cùng vị trí ngẫu nhiên cho mọi biến thể, để so sánh cặp);
3. chạy classifier một lượt cho cả video, reshape thành `(số segment, 19, 10)`;
4. lưu `crops`, `random`, `labels`, `starts`, `ends` vào `outputs_v2/logits/{variant}_{split}.pkl`.

Các bước thực nghiệm sau chỉ đọc file cache này nên rất nhanh, không cần chạy lại mô hình.

In [9]:
def main():
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--variants", default=",".join(FEATURE_VARIANTS),
                    help="vd: legacy,timefix  hoặc  timefix_start,timefix_end (TTA không gian)")
    variants = ap.parse_args().variants.split(",")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = load_baseline(device)
    labels, _ = clean_labels(load_labels("OOPS"))
    split = load_split("OOPS")
    labels["split"] = labels["path"].map(split)
    out_dir = OUTPUTS / "logits"; out_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(0)
    rand_u = {}  # cùng vị trí ngẫu nhiên cho cả 2 biến thể -> so sánh cặp

    for variant in variants:
        for sp in ("train", "val", "test"):
            result = {}
            for path, g in tqdm(labels[labels.split == sp].groupby("path", sort=True), desc=f"{variant}/{sp}"):
                g = g.sort_values("start")
                feats = np.load(feature_dir("OOPS", variant) / f"{Path(path).name}.npy").astype(np.float32)
                seqs, n = [], len(feats)
                for i, (s, e) in enumerate(zip(g.start, g.end)):
                    key = (path, i)
                    if key not in rand_u:
                        rand_u[key] = rng.random(N_RANDOM)
                    for u in list(crop_positions(N_CROPS)) + list(rand_u[key]):
                        seqs.append(gather(feats, window_start(s, e, u, n)))
                logits = predict_logits(model, torch.from_numpy(np.stack(seqs))).numpy()
                logits = logits.reshape(len(g), N_CROPS + N_RANDOM, -1)
                result[path] = {
                    "crops": logits[:, :N_CROPS], "random": logits[:, N_CROPS:],
                    "labels": g.label.to_numpy(), "starts": g.start.to_numpy(), "ends": g.end.to_numpy(),
                }
            with open(out_dir / f"{variant}_{sp}.pkl", "wb") as f:
                pickle.dump(result, f)
            print(f"{variant}/{sp}: {len(result)} video, {sum(len(v['labels']) for v in result.values())} segment")

## ▶ Chạy thật trên dữ liệu thật

Phần trên định nghĩa code. Từ đây là các ô chạy thật bằng dữ liệu, checkpoint và GPU của dự án.

Cắt cửa sổ 18 vector cho từng đoạn của một video thật, chạy mô hình gốc và đối chiếu với logits đã lưu.

In [10]:
import pickle
import numpy as np, pandas as pd, torch
from tcfall.baseline import load_baseline, predict_logits
from tcfall.config import CLASS_NAMES, OUTPUTS, feature_dir
from tcfall.data import clean_labels, load_labels, load_split

labels, _ = clean_labels(load_labels("OOPS"))
split = load_split("OOPS")
test_paths = sorted(p for p in labels.path.unique() if split.get(p) == "test")
path = next(p for p in test_paths if {1, 2} <= set(labels[labels.path == p].label))
g = labels[labels.path == path].sort_values("start")
feats = np.load(feature_dir("OOPS", "timefix") / f"{Path(path).name}.npy").astype(np.float32)
print("Feature của video:", feats.shape, "| vị trí crop trong đoạn:", crop_positions(5).round(3))
model = load_baseline("cuda" if torch.cuda.is_available() else "cpu")
rows = []
for s, e, y in zip(g.start, g.end, g.label):
    seqs = np.stack([gather(feats, window_start(s, e, float(u), len(feats))) for u in crop_positions(5)])
    p = torch.softmax(predict_logits(model, torch.from_numpy(seqs)), -1).mean(0).numpy()
    rows.append({"từ giây": round(s, 2), "đến giây": round(e, 2), "nhãn thật": CLASS_NAMES[y], "dự đoán": CLASS_NAMES[int(p.argmax())], "độ tin": round(float(p.max()), 3)})
print(pd.DataFrame(rows).to_string(index=False))
cache = pickle.load(open(OUTPUTS / "logits" / "timefix_test.pkl", "rb"))[path]
print("Logits đã lưu:", cache["crops"].shape, "= (số đoạn, số crop, 10 lớp)")

Feature của video: (44, 768) | vị trí crop trong đoạn: [0.   0.25 0.5  0.75 1.  ]


 từ giây  đến giây nhãn thật dự đoán  độ tin
    0.00      2.35  standing    fall   0.454
    2.35      3.83      fall    walk   0.560
    3.83      7.81    fallen    walk   0.684
Logits đã lưu: (3, 9, 10) = (số đoạn, số crop, 10 lớp)
